# Nemotron-3-Nano-30B — LoRA Training on Colab

Single-GPU LoRA fine-tuning for the **[NVIDIA Nemotron Model Reasoning Challenge](https://www.kaggle.com/competitions/nvidia-nemotron-model-reasoning-challenge)**.

**Before running:**
1. Set runtime to **GPU** (Runtime → Change runtime type → A100 or L4)
2. Upload `sft_train.jsonl` and `sft_val.jsonl` to **Google Drive** at:
   ```
   My Drive / nemotron-kaggle / data / sft_train.jsonl
   My Drive / nemotron-kaggle / data / sft_val.jsonl
   ```
   Generate these locally with `python src/data/format_sft.py`

| GPU | VRAM | Default Mode | Batch Size | Effective Batch | Est. Time (2 epochs) |
|-----|------|-------------|------------|----------------|---------------------|
| A100-80GB | 80 GB | QLoRA (4-bit) | 8 | 32 | ~4-6 hours |
| RTX PRO 6000 | 96 GB | bf16 (full) | 4 | 32 | ~6-8 hours |

In [2]:
!pip install -U transformers>=4.57.0 datasets accelerate peft trl \
    bitsandbytes safetensors sentencepiece "protobuf<6" huggingface_hub

# ── Mamba-SSM build setup ─────────────────────────────────────
# The model is a Mamba-Transformer hybrid and needs causal-conv1d
# + mamba-ssm, which compile CUDA kernels from source.
import os, torch

os.environ["CUDA_HOME"] = "/usr/local/cuda"
os.environ["MAX_JOBS"] = "4"

# Only compile for *this* GPU — cuts build from ~15 min to ~3 min.
cc_major, cc_minor = torch.cuda.get_device_capability(0)
arch = f"{cc_major}.{cc_minor}"
os.environ["TORCH_CUDA_ARCH_LIST"] = arch
print(f"Building for GPU compute capability {arch} only")

!pip install ninja packaging
!pip install causal-conv1d>=1.6.0 --no-build-isolation --no-cache-dir
!pip install mamba-ssm>=2.3.0   --no-build-isolation --no-cache-dir

# Verify
import importlib
for pkg in ["causal_conv1d", "mamba_ssm"]:
    try:
        m = importlib.import_module(pkg)
        print(f"  ✓ {pkg} {getattr(m, '__version__', 'ok')}")
    except ImportError as e:
        print(f"  ✗ {pkg}: {e}")

Building for GPU compute capability 12.0 only
  ✓ causal_conv1d 1.6.1
  ✓ mamba_ssm 2.3.1


## Configuration

GPU type is auto-detected. Override any setting in this cell before running the rest.

In [3]:
import os
import json
import shutil
import torch

# ── GPU detection ──────────────────────────────────────────────
assert torch.cuda.is_available(), "No GPU found — set Runtime → GPU"
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {GPU_NAME}  ({GPU_MEM_GB:.0f} GB)")

# ── Model ───────────────────────────────────────────────────────
HF_MODEL_ID = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

# ── LoRA (rank ≤ 32 per competition rules) ─────────────────────
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# ── Training hyper-parameters ──────────────────────────────────
NUM_EPOCHS = 2
LEARNING_RATE = 1.5e-4
MAX_SEQ_LENGTH = 2560
WARMUP_RATIO = 0.05

# ── Auto-tune per GPU ──────────────────────────────────────────
# QLoRA model ≈ 18 GB → ~58 GB free for batching (~3.5 GB/sample)
# bf16  model ≈ 60 GB → ~33 GB free on 96 GB   (~3.5 GB/sample)
if GPU_MEM_GB >= 90:          # RTX PRO 6000 (96 GB)
    USE_4BIT = False
    BATCH_SIZE = 4
    GRAD_ACCUM = 8            # effective batch = 32
elif GPU_MEM_GB >= 75:        # A100 (80 GB)
    USE_4BIT = True
    BATCH_SIZE = 8            # safe; try 10 for effective=30
    GRAD_ACCUM = 4            # effective batch = 32
else:                         # smaller GPUs
    USE_4BIT = True
    BATCH_SIZE = 2
    GRAD_ACCUM = 15

# ── Paths ──────────────────────────────────────────────────────
DRIVE_ROOT = "/content/drive/MyDrive/nemotron-kaggle"
DATA_DIR   = os.path.join(DRIVE_ROOT, "data")
DRIVE_OUT  = os.path.join(DRIVE_ROOT, "outputs")
LOCAL_OUT  = "/content/outputs/sft_lora"

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print(f"Mode:  {'QLoRA (4-bit)' if USE_4BIT else 'bf16 (full precision)'}")
print(f"Batch: {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM} effective")
print(f"Epochs: {NUM_EPOCHS}  LR: {LEARNING_RATE}  MaxSeq: {MAX_SEQ_LENGTH}")

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition  (102 GB)
Mode:  bf16 (full precision)
Batch: 4 × 8 = 32 effective
Epochs: 2  LR: 0.00015  MaxSeq: 2560


In [4]:
from google.colab import drive
drive.mount("/content/drive")

for d in [DATA_DIR, DRIVE_OUT, LOCAL_OUT]:
    os.makedirs(d, exist_ok=True)

REPO_RAW = "https://raw.githubusercontent.com/atang-ncat/nvidia-nemotron-reasoning-kaggle/main/data"

for name in ["sft_train.jsonl", "sft_val.jsonl"]:
    path = os.path.join(DATA_DIR, name)
    if os.path.exists(path):
        n = sum(1 for _ in open(path))
        print(f"  ✓ {name}: {n:,} examples")
    else:
        print(f"  ↓ {name}: downloading from GitHub ...")
        !wget -q -O "{path}" "{REPO_RAW}/{name}"
        n = sum(1 for _ in open(path))
        print(f"  ✓ {name}: {n:,} examples (downloaded)")

Mounted at /content/drive
  ✓ sft_train.jsonl: 25,060 examples
  ✓ sft_val.jsonl: 1,319 examples


## Download Model

Downloads the base model from HuggingFace (~60 GB, takes ~10 min on Colab Pro).
Cached on the Colab instance disk for the duration of the session.

In [5]:
from huggingface_hub import snapshot_download

MODEL_CACHE = "/content/model_cache"
print(f"Downloading {HF_MODEL_ID} ...")
MODEL_PATH = snapshot_download(
    HF_MODEL_ID,
    cache_dir=MODEL_CACHE,
    ignore_patterns=["*.gguf", "*.ggml"],
)
print(f"✓ Model ready: {MODEL_PATH}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 35 files:   0%|          | 0/35 [00:00<?, ?it/s]

✓ Model ready: /content/model_cache/models--nvidia--NVIDIA-Nemotron-3-Nano-30B-A3B-BF16/snapshots/cbd3fa9f933d55ef16a84236559f4ee2a0526848


## Load Model + LoRA + Data

Loads everything into GPU memory. Re-run this cell if you change LoRA or data config.

In [6]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# ── Tokenizer ──────────────────────────────────────────────────
print("[1/4] Loading tokenizer ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# ── Model ──────────────────────────────────────────────────────
mode = "4-bit QLoRA" if USE_4BIT else "bf16"
print(f"[2/4] Loading model ({mode}) ...")

load_kwargs = dict(
    pretrained_model_name_or_path=MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",
)
if USE_4BIT:
    load_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
else:
    load_kwargs["torch_dtype"] = torch.bfloat16

model = AutoModelForCausalLM.from_pretrained(**load_kwargs)
model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)
if USE_4BIT:
    model = prepare_model_for_kbit_training(model)

alloc_gb = torch.cuda.memory_allocated(0) / 1e9
print(f"  GPU memory: {alloc_gb:.1f} GB / {GPU_MEM_GB:.0f} GB")

# ── LoRA ────────────────────────────────────────────────────────
print(f"[3/4] Applying LoRA (rank={LORA_R}) ...")
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
trainable, total = model.get_nb_trainable_parameters()
print(f"  Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

# ── Data ─────────────────────────────────────────────────────────
print("[4/4] Loading SFT data ...")

def load_sft_data(split):
    path = os.path.join(DATA_DIR, f"sft_{split}.jsonl")
    texts = []
    with open(path) as f:
        for line in f:
            record = json.loads(line)
            text = tokenizer.apply_chat_template(
                record["messages"],
                tokenize=False,
                add_generation_prompt=False,
            )
            texts.append({"text": text})
    return Dataset.from_list(texts)

train_dataset = load_sft_data("train")
val_dataset   = load_sft_data("val")
print(f"  Train: {len(train_dataset):,}    Val: {len(val_dataset):,}")
print("\n✓ Ready to train!")

[1/4] Loading tokenizer ...


`torch_dtype` is deprecated! Use `dtype` instead!


[2/4] Loading model (bf16) ...


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

  GPU memory: 63.2 GB / 102 GB
[3/4] Applying LoRA (rank=32) ...
  Trainable: 869,318,656 / 32,447,256,000 (2.68%)
[4/4] Loading SFT data ...
  Train: 25,060    Val: 1,319

✓ Ready to train!


## Train

Checkpoints are saved locally and backed up to Google Drive after each epoch.
If the session disconnects, re-run all cells above — training resumes from the
last Drive checkpoint automatically.

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback

class DriveBackupCallback(TrainerCallback):
    """Copy checkpoint to Drive after each save for session resilience."""
    def on_save(self, args, state, control, **kwargs):
        src = os.path.join(args.output_dir, f"checkpoint-{state.global_step}")
        dst = os.path.join(DRIVE_OUT, f"checkpoint-{state.global_step}")
        if os.path.isdir(src):
            if os.path.isdir(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            # Clean old Drive checkpoints (keep latest 2)
            drive_ckpts = sorted(
                [d for d in os.listdir(DRIVE_OUT) if d.startswith("checkpoint-")],
                key=lambda x: int(x.split("-")[1]),
            )
            for old in drive_ckpts[:-2]:
                shutil.rmtree(os.path.join(DRIVE_OUT, old), ignore_errors=True)
            print(f"  ↳ Backed up to Drive: {dst}")

training_args = SFTConfig(
    output_dir=LOCAL_OUT,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=max(1, BATCH_SIZE // 2),
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    weight_decay=0.01,
    bf16=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    max_length=MAX_SEQ_LENGTH,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
    dataloader_num_workers=2,
    remove_unused_columns=False,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    callbacks=[DriveBackupCallback()],
)

# ── Resume from checkpoint (local first, then Drive) ────────────
resume_ckpt = None
for search_dir in [LOCAL_OUT, DRIVE_OUT]:
    if os.path.isdir(search_dir):
        ckpts = sorted(
            [d for d in os.listdir(search_dir) if d.startswith("checkpoint-")],
            key=lambda x: int(x.split("-")[1]),
        )
        if ckpts:
            candidate = os.path.join(search_dir, ckpts[-1])
            if resume_ckpt is None:
                resume_ckpt = candidate
            else:
                cur_step = int(os.path.basename(resume_ckpt).split("-")[1])
                new_step = int(ckpts[-1].split("-")[1])
                if new_step > cur_step:
                    resume_ckpt = candidate

if resume_ckpt:
    print(f"Resuming from checkpoint: {resume_ckpt}")

# ── Train ──────────────────────────────────────────────────────
eff = BATCH_SIZE * GRAD_ACCUM
print(f"\n{'='*60}")
print(f"  TRAINING: {NUM_EPOCHS} epochs, batch {BATCH_SIZE}×{GRAD_ACCUM}={eff}")
print(f"  LR: {LEARNING_RATE}  Seq: {MAX_SEQ_LENGTH}  GPU: {GPU_NAME}")
print(f"{'='*60}\n")

trainer.train(resume_from_checkpoint=resume_ckpt)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/25060 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/25060 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/1319 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1319 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 11, 'pad_token_id': 11}.



  TRAINING: 2 epochs, batch 4×8=32
  LR: 0.00015  Seq: 2560  GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition



`use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss


## Save & Export

Saves the LoRA adapter to Google Drive and creates a `submission.zip` ready for Kaggle upload.

In [ ]:
import zipfile

# ── Save adapter ──────────────────────────────────────────────
adapter_local = os.path.join(LOCAL_OUT, "final_adapter")
model.save_pretrained(adapter_local)
tokenizer.save_pretrained(adapter_local)

with open(os.path.join(adapter_local, "adapter_config.json")) as f:
    rank = json.load(f).get("r", -1)
print(f"LoRA rank: {rank}  (competition max: 32)")
assert int(rank) <= 32, f"Rank {rank} exceeds competition limit!"

# ── Copy to Drive ─────────────────────────────────────────────
adapter_drive = os.path.join(DRIVE_OUT, "final_adapter")
if os.path.isdir(adapter_drive):
    shutil.rmtree(adapter_drive)
shutil.copytree(adapter_local, adapter_drive)
print(f"✓ Adapter on Drive: {adapter_drive}")

# ── Create submission zip ─────────────────────────────────────
zip_path = os.path.join(DRIVE_OUT, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(adapter_local):
        for fname in files:
            fpath = os.path.join(root, fname)
            arcname = os.path.relpath(fpath, adapter_local)
            zf.write(fpath, arcname)

zip_mb = os.path.getsize(zip_path) / 1e6
print(f"✓ Submission zip: {zip_path}  ({zip_mb:.1f} MB)")
print("  Upload this file to Kaggle as your competition submission.")

In [ ]:
# Download the submission zip to your local machine
from google.colab import files
files.download(zip_path)